# Tutorial 04: Creating a Time-Evolution Video

This tutorial shows how to create an MP4 video showing how multiple wing patterns evolve over time.

The workflow consists of three stages:

1. **Batch solve with periodic snapshots** -- run the reaction-diffusion simulation for a population of morphs, saving pattern images at regular intervals.
2. **Merge snapshots into composite frames** -- for each saved timestep, combine the individual morph images into a single composite image showing all morphs side by side.
3. **Encode frames into video** -- stitch the composite frames together into an MP4 video file.

## 1. Setup

Import the required libraries. We need `lpf.data` to load pre-defined model parameters, `lpf.initializers` and `lpf.models` for the Liaw reaction-diffusion model, and `lpf.solvers` for the numerical solver.

In [ ]:
import os
import os.path as osp
from os.path import join as pjoin
import time
from datetime import datetime

import numpy as np
np.seterr(all='raise')

from lpf.data import load_model_dicts
from lpf.initializers import LiawInitializer
from lpf.models import LiawModel
from lpf.solvers import EulerSolver

## 2. Simulation Parameters

Configure the device, time-stepping parameters, and spatial grid dimensions. Set `device` to `"cpu"` if no GPU is available.

In [ ]:
# Select device: CPU or GPU.
device = "cuda:0"

# Time parameters
dt = 0.01
n_iters = 500000

# Space parameters
dx = 0.1
width = 128
height = 128
shape = (height, width)

Create a timestamped output directory so that each experiment run is stored separately.

In [ ]:
# Create the output directory.
str_now = datetime.now().strftime('%Y%m%d-%H%M%S')
dpath_output = pjoin(osp.abspath("./output"), "experiment_batch_%s" % (str_now))
os.makedirs(dpath_output, exist_ok=True)

## 3. Load Population

Load a population of pre-defined model parameter sets from JSON files. Each file in the population directory contains the kinetic parameters, initial conditions, and color information for one morph. The batch simulation will run all of them simultaneously.

In [ ]:
# Load a population of previously defined models.
LPF_REPO_HOME = r"D:/repos/lpf"
dpath_pop = pjoin(LPF_REPO_HOME, "population", "test_pop_01")
model_dicts = load_model_dicts(dpath_pop)

## 4. Model Construction and Solve

Build the batch Liaw model from the loaded population parameters and run the simulation. The initializer sets up the initial concentration fields, and `LiawModel.parse_params` extracts the kinetic parameters from the model dictionaries.

The solver is configured with `period_output=10000`, which saves morph and pattern images every 10,000 iterations. This produces a series of snapshots that capture the time evolution of each morph's pattern.

In [ ]:
# Create the Liaw initializer.
initializer = LiawInitializer()
initializer.update(model_dicts)
params = LiawModel.parse_params(model_dicts)

In [ ]:
# Create the Liaw model.
model = LiawModel(initializer=initializer, params=params, width=width, height=height, dx=dx, device=device)

In [ ]:
# Create the Euler solver.
solver = EulerSolver()

t_beg = time.time()

solver.solve(
    model=model,
    dt=dt,
    n_iters=n_iters,
    period_output=10000,
    dpath_model=dpath_output,
    dpath_morph=dpath_output,
    dpath_pattern=dpath_output,
    verbose=1
)

t_end = time.time()

print("Elapsed time: %f sec." % (t_end - t_beg))

## 5. Create Composite Frames

Import the visualization utilities. The function `merge_multiple_timeseries` scans the per-model output directories, and for each timestep, creates a single composite image that arranges all morphs in a grid. This makes it easy to compare how different parameter sets evolve in parallel.

The `n_cols` parameter controls how many morphs appear per row, `ratio_resize` scales the output images, and `text_format` adds a label to each morph panel.

In [ ]:
from lpf.visualization import merge_multiple_timeseries
from lpf.visualization import create_video

In [ ]:
!ls {dpath_output}

In [ ]:
dpath_frames = osp.join(dpath_output, "frames")
imgs = merge_multiple_timeseries(dpath_input=dpath_output,
                                 dpath_output=dpath_frames,
                                 n_cols=4,
                                 ratio_resize=1.0,
                                 text_format="morph = ",
                                 font_size=16)

In [ ]:
# Check the new "frames" directory
!ls {dpath_output}

## 6. Encode Video

Use `create_video` to combine the composite frame images into an MP4 video file. The `fps` parameter sets the frame rate, and `duration` controls how long each frame is displayed. The resulting video shows the full time evolution of all morphs in the population.

In [ ]:
create_video(dpath_frames, "video_morphs.mp4", fps=32, duration=0.1)

Display the generated video directly in the notebook.

In [ ]:
from IPython.display import Video
Video("video_morphs.mp4")